# Chapter 6 &mdash; A Transformer with Context $k$ is a Finite-State Machine

**Concept 13 of the Chapter 6 decomposition:** *A Transformer with Context $k$ is a Finite-State Machine*

Karpathy's baby GPT has $v^k$ states before it is trained at all &mdash; and pointed at a DFA it learns the acceptance condition exactly.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Transformer-Is-Finite-State/Concept-Transformer-Is-Finite-State.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.GPTLab         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A transformer with context length $k$ over a vocabulary of size $v$ **is** a
finite-state machine with $v^k$ states. Its state is the last $k$ tokens, and that is
settled by the **architecture**, before any training. Andrej Karpathy made the point
with an eight-state GPT; here it is pointed at a DFA.

**The END token earns its place.** A next-symbol model can only express constraints on
*continuations*. Acceptance is a constraint on where you may **stop** &mdash; so
without a token meaning *"the string ends here"* a generator cannot represent a
language like parity at all, since every binary string is a prefix of an accepted one.
With END, $P(\text{END}\mid w)$ is precisely the model's opinion about accepting $w$,
and can be laid beside the DFA's answer.

So the experiment is: train on every accepted string up to a length, then ask the
model, for each of the $2^k$ windows, whether a string may end there &mdash; and
compare with the machine.

*Needs `torch`, which is preinstalled on Colab.*

## 2. Definitions

### The language, and its minimal machine

In [ ]:
# --- the languages we will ask about ------------------------------------
ENDS01 = md2mc('''DFA
I  : 0 -> S0
I  : 1 -> I
S0 : 0 -> S0
S0 : 1 -> F
F  : 0 -> S0
F  : 1 -> I
''')

PARITY = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> S1
S1 : 0 -> S1
S1 : 1 -> IF
''')

DIV3 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> S1
S1 : 0 -> S2
S1 : 1 -> IF
S2 : 0 -> S1
S2 : 1 -> S2
''')

NO11 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> F1
F1 : 0 -> IF
F1 : 1 -> D
D  : 0|1 -> D
''')

print('min-DFA of ENDS01:', len(min_dfa(ENDS01)['Q']), 'states')
dotObj_dfa(min_dfa(ENDS01), FuseEdges=True)

### The corpus: accepted strings, each closed by END

In [ ]:
from jove.GPTLab import *

acc, seq = corpus(ENDS01)
print('accepted strings :', len(acc))
print('first few        :', [s or 'eps' for s in acc[:6]])
print('corpus tokens    :', len(seq), ' (2 means END)')
print('as tokens        :', seq[:24], '...')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;12.&nbsp;DeMorgan's Law for DFA, Verified by Isomorphism](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-DeMorgan-For-DFA/Concept-DeMorgan-For-DFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;14.&nbsp;Which DFAs a $k$-Window Can Learn: Myhill-Nerode Against a Window](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Which-DFAs-A-Window-Can-Learn/Concept-Which-DFAs-A-Window-Can-Learn.ipynb)&nbsp;&rarr;

---

## 3. Tests

**The state count is fixed before training.** That is the whole claim.

In [ ]:
for k in (2, 3, 4):
    print('  context %d  ->  %d**%d = %3d states' % (k, 3, k, 3 ** k))
print()
print('Over {0,1,END} a context-3 model has 27 states whatever it is shown.')
print('Only 8 of them are windows of actual symbols, and those are the ones')
print('we interrogate below.')

**Train it.** A few seconds; Karpathy's model is tiny.

In [ ]:
ok, why = torch_available()
if ok:
    g = train(seq, k=3)
    print('trained on %d examples, final loss %.4f' % (g.examples, g.final_loss))
else:
    print('torch is not available here:', why)
    print('On Colab this trains in a few seconds.')

**What it learned.** `P(END)` against what the DFA accepts.

In [ ]:
if ok:
    sep, margin, rows = report(g, ENDS01, k=3, seen=windows_seen(seq, 3))
    assert sep, 'expected P(END) to separate accepted from rejected'
else:
    print('(on Colab: P(END) is about 0.6 on exactly 001 and 101 -- the')
    print(' windows ending in 01 -- and about 0.00 on the other six, so')
    print(' the two groups separate with a margin of about 0.55.)')

The learned machine and the minimal DFA are answering the same question.

In [ ]:
print('The DFA accepts a string iff it ends in 01.')
print('The model puts its END probability on exactly the windows ending 01.')
print()
print('Those are not the same object -- the DFA has 3 states and the model')
print('has 8 -- but on this language they induce the same partition of')
print('strings into accepted and rejected.  The next concept asks when that')
print('can happen at all.')

## 4. Exercises


1. Re-run with `k=2`. Does it still agree on every window? What about `k=1`?
2. `NO11` (no two consecutive 1s) is defined above. Predict `P(END)` for each window
   before training, then check.
3. The corpus holds every accepted string up to a length. Halve it and retrain. Which
   windows degrade first, and why those?
4. A window of three symbols gives 8 states, but the model really has $3^3 = 27$.
   What are the other 19, and why does no training data ever put it in them?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Transformer-Is-Finite-State')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')